In [ ]:
# ======================================================================
# SHONA CORPUS STOP-WORD EXTRACTION
# ======================================================================
#
# PURPOSE
# -------
# Extract frequent words from the collected Shona X/Twitter corpus and
# identify candidate stop words for manual linguistic validation.
#
# INPUT
# -----
# shona_x_corpus_50000_full_archive.csv
#
# OUTPUT
# ------
# shona_word_frequency.csv
# shona_candidate_stopwords.csv
# shona_stopwords_review.xlsx
# shona_stopwords.txt
#
# IMPORTANT
# ---------
# Frequency alone does NOT make a word a stop word.
# Sentiment-bearing and domain-specific words must be retained.
# ======================================================================

import os
import re
from collections import Counter

import pandas as pd


# ======================================================================
# CONFIGURATION
# ======================================================================

from google.colab import files

uploaded = files.upload()

import os

print(os.listdir('/content'))
INPUT_FILE = "shona_x_corpus_50000_full_archive.csv"

WORD_FREQUENCY_FILE = "shona_word_frequency.csv"
CANDIDATE_FILE = "shona_candidate_stopwords.csv"
EXCEL_FILE = "shona_stopwords_review.xlsx"
STOPWORDS_FILE = "shona_stopwords.txt"

# Minimum number of occurrences in the complete corpus
MIN_FREQUENCY = 20

# Minimum percentage of documents containing the word
DOCUMENT_FREQUENCY_THRESHOLD = 0.10


# ======================================================================
# KNOWN SHONA FUNCTION WORDS
# ======================================================================
#
# These are only seed candidates.
# The corpus-derived list must still be manually validated.
# ======================================================================

SHONA_FUNCTION_WORDS = {
    "uye",
    "asi",
    "kana",
    "kuti",
    "kutiwo",
    "se",
    "saka",
    "nekuti",
    "nokuti",
    "zvino",
    "zvichida",
    "zvakare",
    "zvokuti",
    "pakati",
    "pasi",
    "pamusoro",
    "mukati",
    "kunze",
    "pano",
    "ipapo",
    "apo",
    "uko",
    "kuno",
    "uko",
    "ini",
    "iwe",
    "isu",
    "imi",
    "ivo",
    "iye",
    "ndini",
    "ndiye",
    "ndivo",
    "ndiko",
    "ndipo",
    "ndiko",
    "chii",
    "ani",
    "sei",
    "rini",
    "chimwe",
    "zvimwe",
    "wese",
    "vese",
    "ose",
    "ese",
    "ino",
    "icho",
    "izvi",
    "izvo",
    "uyu",
    "uyuwo",
    "ava",
    "avo",
    "avawo",
    "kana",
    "kana kuti",
}


# ======================================================================
# COMMON ENGLISH FUNCTION WORDS
# ======================================================================
#
# These are included because the corpus may contain Shona-English
# code-switching.
# ======================================================================

COMMON_ENGLISH = {
    "the",
    "a",
    "an",
    "and",
    "or",
    "but",
    "if",
    "then",
    "than",
    "of",
    "to",
    "in",
    "on",
    "at",
    "for",
    "from",
    "with",
    "by",
    "as",
    "is",
    "am",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "it",
    "this",
    "that",
    "these",
    "those",
    "he",
    "she",
    "they",
    "we",
    "you",
    "i",
    "me",
    "my",
    "our",
    "your",
    "their",
    "his",
    "her",
    "them",
    "do",
    "does",
    "did",
    "not",
    "no",
    "yes",
    "can",
    "could",
    "will",
    "would",
    "should",
    "may",
    "might",
    "must",
}


# ======================================================================
# SENTIMENT WORDS THAT SHOULD NOT AUTOMATICALLY BECOME STOP WORDS
# ======================================================================

SENTIMENT_PROTECTED = {
    "rudo",
    "kuvenga",
    "kuvengwa",
    "kufara",
    "mufaro",
    "kusuwa",
    "kusuruvara",
    "hasha",
    "kutsamwa",
    "kutya",
    "kutambura",
    "kurwadziwa",
    "kurwara",
    "hutano",
    "chirwere",
    "rufu",
    "kufa",
    "dambudziko",
    "matambudziko",
    "huori",
    "hurombo",
    "nzara",
    "mhirizhonga",
    "kunaka",
    "zvakaipa",
    "zvakanaka",
    "zvakaipisisa",
    "zvinoshamisa",
    "kubudirira",
    "kukundikana",
    "kukundwa",
    "kukunda",
}


# ======================================================================
# DOMAIN WORDS THAT SHOULD NOT AUTOMATICALLY BECOME STOP WORDS
# ======================================================================

DOMAIN_PROTECTED = {
    "chipatara",
    "zvipatara",
    "chiremba",
    "vanachiremba",
    "murwere",
    "varwere",
    "hutano",
    "chirwere",
    "dzidzo",
    "chikoro",
    "zvikoro",
    "mudzidzi",
    "vadzidzi",
    "mudzidzisi",
    "vadzidzisi",
    "hurumende",
    "mutungamiri",
    "vatungamiri",
    "paramende",
    "sarudzo",
    "bato",
    "mapato",
    "nyika",
    "mutemo",
    "mitemo",
    "hupfumi",
    "mari",
    "basa",
    "mabasa",
}


# ======================================================================
# TEXT CLEANING
# ======================================================================

def clean_text(text):
    """
    Clean tweet text while preserving words.
    """

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove mentions
    text = re.sub(r"@\w+", " ", text)

    # Remove hashtag symbol but retain the hashtag word
    text = re.sub(r"#", "", text)

    # Replace punctuation with spaces
    text = re.sub(r"[^\w\s']", " ", text, flags=re.UNICODE)

    # Remove isolated numbers
    text = re.sub(r"\b\d+\b", " ", text)

    # Remove repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ======================================================================
# TOKENIZATION
# ======================================================================

def tokenize(text):
    """
    Convert cleaned text into word tokens.
    """

    text = clean_text(text)

    if not text:
        return []

    tokens = re.findall(
        r"\b[\w']+\b",
        text,
        flags=re.UNICODE
    )

    return [
        token.lower()
        for token in tokens
        if len(token) > 1
    ]


# ======================================================================
# LOAD DATASET
# ======================================================================

print("=" * 70)
print("SHONA CORPUS STOP-WORD EXTRACTION")
print("=" * 70)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"\nInput file not found:\n{INPUT_FILE}\n\n"
        "Place the CSV file in the same folder as this Python script."
    )

print(f"\nLoading corpus:")
print(INPUT_FILE)

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print(f"Rows loaded: {len(df):,}")


# ======================================================================
# IDENTIFY TEXT COLUMN
# ======================================================================

if "cleaned_text" in df.columns:
    TEXT_COLUMN = "cleaned_text"

elif "text" in df.columns:
    TEXT_COLUMN = "text"

else:
    raise ValueError(
        "No suitable text column found. "
        "Expected 'cleaned_text' or 'text'."
    )

print(f"Text column used: {TEXT_COLUMN}")


# ======================================================================
# TOKEN FREQUENCY
# ======================================================================

word_frequency = Counter()

document_frequency = Counter()

total_documents = 0

print("\nProcessing corpus...")

for text in df[TEXT_COLUMN].fillna(""):

    tokens = tokenize(text)

    if not tokens:
        continue

    total_documents += 1

    # Count every occurrence
    word_frequency.update(tokens)

    # Count each word only once per document
    unique_tokens = set(tokens)

    document_frequency.update(unique_tokens)


print(f"Documents processed: {total_documents:,}")
print(f"Unique words found: {len(word_frequency):,}")


# ======================================================================
# CREATE FREQUENCY TABLE
# ======================================================================

frequency_data = []

for word, frequency in word_frequency.most_common():

    doc_freq = document_frequency[word]

    if total_documents > 0:
        document_ratio = doc_freq / total_documents
    else:
        document_ratio = 0

    frequency_data.append({
        "word": word,
        "frequency": frequency,
        "document_frequency": doc_freq,
        "document_frequency_ratio": round(
            document_ratio,
            6
        )
    })


frequency_df = pd.DataFrame(frequency_data)


# ======================================================================
# SAVE COMPLETE WORD FREQUENCY LIST
# ======================================================================

frequency_df.to_csv(
    WORD_FREQUENCY_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nSaved complete word-frequency list:"
    f"\n{WORD_FREQUENCY_FILE}"
)


# ======================================================================
# IDENTIFY CANDIDATE STOP WORDS
# ======================================================================

candidate_rows = []

for _, row in frequency_df.iterrows():

    word = row["word"]
    frequency = int(row["frequency"])
    doc_frequency = int(row["document_frequency"])
    doc_ratio = float(row["document_frequency_ratio"])

    # --------------------------------------------------------------
    # Basic frequency filter
    # --------------------------------------------------------------

    if frequency < MIN_FREQUENCY:
        continue

    if doc_ratio < DOCUMENT_FREQUENCY_THRESHOLD:
        continue

    # --------------------------------------------------------------
    # Determine category
    # --------------------------------------------------------------

    if word in SENTIMENT_PROTECTED:
        category = "SENTIMENT_WORD"

    elif word in DOMAIN_PROTECTED:
        category = "DOMAIN_WORD"

    elif word in SHONA_FUNCTION_WORDS:
        category = "KNOWN_SHONA_FUNCTION_WORD"

    elif word in COMMON_ENGLISH:
        category = "ENGLISH_FUNCTION_WORD"

    else:
        category = "CORPUS_FREQUENT_CANDIDATE"

    # --------------------------------------------------------------
    # Candidate decision
    # --------------------------------------------------------------

    if category in {
        "SENTIMENT_WORD",
        "DOMAIN_WORD"
    }:
        recommendation = "KEEP"

    elif category == "KNOWN_SHONA_FUNCTION_WORD":
        recommendation = "REVIEW_AS_STOP_WORD"

    elif category == "ENGLISH_FUNCTION_WORD":
        recommendation = "REVIEW_CODE_SWITCHING"

    else:
        recommendation = "MANUAL_REVIEW"

    candidate_rows.append({
        "word": word,
        "frequency": frequency,
        "document_frequency": doc_frequency,
        "document_frequency_ratio": doc_ratio,
        "category": category,
        "recommendation": recommendation
    })


candidate_df = pd.DataFrame(candidate_rows)


# ======================================================================
# SAVE CANDIDATE LIST
# ======================================================================

candidate_df.to_csv(
    CANDIDATE_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nSaved candidate stop-word list:"
    f"\n{CANDIDATE_FILE}"
)


# ======================================================================
# CREATE REVIEW EXCEL WORKBOOK
# ======================================================================

known_shona_rows = []

for word in sorted(SHONA_FUNCTION_WORDS):

    if word in word_frequency:

        known_shona_rows.append({
            "word": word,
            "frequency": word_frequency[word],
            "document_frequency": document_frequency[word],
            "recommendation": "REVIEW_AS_STOP_WORD"
        })

    else:

        known_shona_rows.append({
            "word": word,
            "frequency": 0,
            "document_frequency": 0,
            "recommendation": "NOT_FOUND_IN_CORPUS"
        })


known_shona_df = pd.DataFrame(
    known_shona_rows
)


# ----------------------------------------------------------------------
# Sentiment protected words
# ----------------------------------------------------------------------

sentiment_rows = []

for word in sorted(SENTIMENT_PROTECTED):

    sentiment_rows.append({
        "word": word,
        "frequency": word_frequency.get(word, 0),
        "document_frequency": document_frequency.get(word, 0),
        "recommendation": "KEEP"
    })


sentiment_df = pd.DataFrame(
    sentiment_rows
)


# ----------------------------------------------------------------------
# Domain protected words
# ----------------------------------------------------------------------

domain_rows = []

for word in sorted(DOMAIN_PROTECTED):

    domain_rows.append({
        "word": word,
        "frequency": word_frequency.get(word, 0),
        "document_frequency": document_frequency.get(word, 0),
        "recommendation": "KEEP"
    })


domain_df = pd.DataFrame(
    domain_rows
)


# ----------------------------------------------------------------------
# Write Excel workbook
# ----------------------------------------------------------------------

with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    frequency_df.to_excel(
        writer,
        sheet_name="Word Frequency",
        index=False
    )

    candidate_df.to_excel(
        writer,
        sheet_name="Candidate Stopwords",
        index=False
    )

    known_shona_df.to_excel(
        writer,
        sheet_name="Known Shona Candidates",
        index=False
    )

    sentiment_df.to_excel(
        writer,
        sheet_name="Protected Sentiment",
        index=False
    )

    domain_df.to_excel(
        writer,
        sheet_name="Protected Domain",
        index=False
    )


print(
    f"\nSaved Excel review workbook:"
    f"\n{EXCEL_FILE}"
)


# ======================================================================
# CREATE INITIAL SHONA STOP-WORD FILE
# ======================================================================
#
# Only words already identified as Shona function words are included.
# Corpus-frequent words are deliberately NOT automatically added.
# ======================================================================

final_seed_stopwords = sorted(
    set(SHONA_FUNCTION_WORDS)
    - set(SENTIMENT_PROTECTED)
    - set(DOMAIN_PROTECTED)
)


with open(
    STOPWORDS_FILE,
    "w",
    encoding="utf-8"
) as f:

    for word in final_seed_stopwords:
        f.write(word + "\n")


print(
    f"\nSaved initial Shona stop-word file:"
    f"\n{STOPWORDS_FILE}"
)


# ======================================================================
# DISPLAY TOP WORDS
# ======================================================================

print("\n" + "=" * 70)
print("TOP 50 MOST FREQUENT WORDS")
print("=" * 70)

print(
    frequency_df.head(50).to_string(
        index=False
    )
)


# ======================================================================
# DISPLAY CANDIDATE STOP WORDS
# ======================================================================

print("\n" + "=" * 70)
print("CANDIDATE STOP WORDS")
print("=" * 70)

if len(candidate_df) > 0:

    print(
        candidate_df[
            [
                "word",
                "frequency",
                "document_frequency_ratio",
                "category",
                "recommendation"
            ]
        ]
        .head(100)
        .to_string(index=False)
    )

else:

    print(
        "No candidate stop words were identified "
        "using the current thresholds."
    )


# ======================================================================
# SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("STOP-WORD EXTRACTION SUMMARY")
print("=" * 70)

print(f"Corpus documents       : {len(df):,}")
print(f"Documents with text    : {total_documents:,}")
print(f"Unique words           : {len(word_frequency):,}")
print(f"Candidate words        : {len(candidate_df):,}")
print(
    f"Known Shona candidates : "
    f"{len(final_seed_stopwords):,}"
)

print("\nOutput files:")
print(f"1. {WORD_FREQUENCY_FILE}")
print(f"2. {CANDIDATE_FILE}")
print(f"3. {EXCEL_FILE}")
print(f"4. {STOPWORDS_FILE}")

print("\nIMPORTANT:")
print(
    "The candidate list must be linguistically validated "
    "before being used in the final sentiment-analysis pipeline."
)

print("\nProcessing completed successfully.")

Saving shona_x_corpus_50000_full_archive.csv to shona_x_corpus_50000_full_archive.csv
['.config', 'shona_x_corpus_50000_full_archive.csv', 'sample_data']
SHONA CORPUS STOP-WORD EXTRACTION

Loading corpus:
shona_x_corpus_50000_full_archive.csv
Rows loaded: 39,666
Text column used: cleaned_text

Processing corpus...
Documents processed: 39,665
Unique words found: 84,397

Saved complete word-frequency list:
shona_word_frequency.csv

Saved candidate stop-word list:
shona_candidate_stopwords.csv

Saved Excel review workbook:
shona_stopwords_review.xlsx

Saved initial Shona stop-word file:
shona_stopwords.txt

TOP 50 MOST FREQUENT WORDS
       word  frequency  document_frequency  document_frequency_ratio
    chikoro      13089               12781                  0.322224
   chiremba      11384               11192                  0.282163
  hurumende       6150                5927                  0.149426
       kuti       5206                4465                  0.112568
        the     